# Modeling the Upper Dimensional-Deviation Tail with PROC QUANTREG

## Executive Summary

A precision-machining plant cares about the **worst-case** part-to-part dimensional deviation, not just the average, because the upper tail drives scrap and customer rejects. This notebook uses **PROC QUANTREG** to fit quantile regressions at the median and the 90th percentile, revealing that machine wear, cycle speed, and feed rate exert a much stronger pull on the high-quantile (scrap-risk) tail than on the median — the signature of a heteroscedastic process where things get more variable as the tool wears.

## Data Sources

| Dataset | Rows | Description | Key Variables |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Synthetic part-level records from a CNC turning line, generated inline with `call streaminit`/`rand`. Dimensional deviation (microns) from nominal is modeled with a heteroscedastic error whose spread grows with tool wear and cycle speed, so process drivers act more strongly on the upper tail than on the median. | `Deviation` (response, microns), `ToolWear` (cumulative cut minutes), `CycleSpeed` (rpm), `CoolantTemp` (deg C), `Humidity` (%RH), `FeedRate` (mm/rev), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Modeling Process Drivers of the Upper Dimensional-Deviation Tail

## Manufacturing use case: scrap-risk modeling on a CNC turning line

On a precision-machining line, parts are scrapped when the **dimensional deviation** from nominal grows too large. The plant does not lose money on the *average* part — it loses money on the **upper tail** of the deviation distribution. Ordinary least-squares regression models the conditional mean and can completely miss drivers that only matter when things go wrong.

**Quantile regression** models a chosen conditional quantile (for example the 90th percentile of deviation) instead of the mean. **PROC QUANTREG** fits one or several quantiles in a single call and reports a parameter estimate, standard error, and confidence limits for each driver at each quantile. We will:

1. Generate a realistic synthetic part-level dataset whose error spread grows with tool wear and cycle speed (so drivers hit the tail harder than the center).
2. Fit the **median** (q = 0.50) and the **scrap-risk tail** (q = 0.90) in one PROC QUANTREG call.
3. Compare the two coefficient sets side by side to show how the slopes change from center to tail.
4. Score every part with its fitted 90th-percentile prediction so analysts can flag high-tail-risk parts.

Everything below is self-contained — no external files, no network.

## Step 1 — Generate synthetic machining data

We simulate turned parts across 4 machines and 3 shifts. The key realism trick is **heteroscedasticity**: the standard deviation of the random error term (`sigma`) grows with `ToolWear` and `CycleSpeed`. That makes those two drivers exert a much stronger pull on the 90th percentile of `Deviation` than on its median — exactly the situation where quantile regression earns its keep. `Humidity` carries no real signal in the data-generating process, so it serves as a placebo driver we can watch.

In [1]:
data work.machining;
    call streaminit(20260531);
    length Machine $2 Shift $5;
    do PartID = 1 to 100;
        /* CLASS factors */
        m = rand('integer', 1, 4);
        Machine = cats('M', put(m, 1.));
        s = rand('integer', 1, 3);
        if s = 1 then Shift = 'Day';
        else if s = 2 then Shift = 'Eve';
        else Shift = 'Night';

        /* Continuous process drivers */
        ToolWear     = rand('uniform') * 120;            /* cumulative cut minutes */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* spindle rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* deg C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (placebo) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rev */

        /* Machine offsets: one machine runs hotter */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Night shift drifts slightly */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Location (central tendency) of deviation in microns */
        mu = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Heteroscedastic spread: tail widens with wear & speed */
        sigma = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = mu + sigma * rand('normal');
        if Deviation < 0 then Deviation = 0;

        keep PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        output;
    end;
run;

NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


### Quick look at the raw distribution

Before modeling, confirm the response is right-skewed with a meaningful upper tail — that is the part of the distribution that drives scrap. We print the median and the upper percentiles directly from PROC UNIVARIATE so we can see how much higher the 90th percentile sits above the median.

In [2]:
proc univariate data=work.machining noprint;
    var Deviation;
    output out=work.devpct
        median=Med p90=P90 p95=P95 p99=P99;
run;

proc print data=work.devpct noobs label;
    label Med = 'Median Deviation'
          P90 = '90th Pctl'
          P95 = '95th Pctl'
          P99 = '99th Pctl';
run;


Median Deviation      90th Pctl      95th Pctl      99th Pctl
----------------  -------------  -------------  -------------
    8.7251490709  12.4132063767  13.5691645665  17.0510365805



NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Step 2 — Fit the median and the scrap-risk tail together

PROC QUANTREG fits both quantiles in a single call: `QUANTILE=0.5 0.90`. The `CLASS` statement declares the categorical process factors (`Machine`, `Shift`); the `MODEL` statement lists all candidate continuous effects. We request `CI=SPARSITY`, which uses the sparsity-function estimator to produce a standard error and 95% confidence band for every coefficient.

Read the two blocks of output as a before/after: the first block (q = 0.50) describes the *typical* part; the second (q = 0.90) describes the *scrap-prone* part. Watch `ToolWear`, `CycleSpeed`, and `FeedRate` — because the simulated error spread grows with wear and speed, their slopes should be larger at the upper quantile.

In [3]:
proc quantreg data=work.machining ci=sparsity;
    class Machine Shift;
    model Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
run;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1

NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Step 3 — Put the center and the tail side by side

Reading two coefficient blocks in parallel is awkward, so we capture the parameter table with `ODS OUTPUT ParameterEstimates=` (which carries a `Quantile` column), then merge the q = 0.50 and q = 0.90 estimates for the continuous drivers into one comparison table. The `Tail - Median` column is the headline number: a large positive value means the driver pushes the scrap-risk tail much harder than it moves the typical part.

In [4]:
ods output ParameterEstimates=work.pe;
proc quantreg data=work.machining ci=sparsity;
    class Machine Shift;
    model Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
run;

/* Merge the median and tail slopes for each continuous driver */
data work.compare;
    keep Parameter MedianSlope TailSlope TailMinusMedian;
    merge
        work.pe(where=(Quantile=0.5)
                keep=Quantile Parameter Estimate
                rename=(Estimate=MedianSlope))
        work.pe(where=(Quantile=0.9)
                keep=Quantile Parameter Estimate
                rename=(Estimate=TailSlope));
    by Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
run;

proc print data=work.compare noobs label;
    label Parameter       = 'Driver'
          MedianSlope     = 'Slope @ q=0.50'
          TailSlope       = 'Slope @ q=0.90'
          TailMinusMedian = 'Tail - Median';
    format MedianSlope TailSlope TailMinusMedian 10.4;
run;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


     Driver  Slope @ 

NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Step 4 — Score every part at the 90th percentile

The `OUTPUT` statement writes the fitted 90th-percentile prediction back out, one row per part, alongside the prediction standard error (`STDP`) and the check-loss residual. We carry the `PartID` through with the `ID` statement and copy the two dominant drivers so analysts can sort the riskiest parts to the top. A small PROC PRINT shows the first scored parts.

In [5]:
proc quantreg data=work.machining ci=sparsity;
    class Machine Shift;
    id PartID;
    model Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    output out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
run;

proc print data=work.scored(obs=10) noobs label;
    var PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    label PredP90 = 'Predicted 90th Pctl Deviation'
          SEPred  = 'Prediction Std Err'
          Resid   = 'Check-Loss Residual';
run;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED  Predicted 90th Pctl Deviation  Prediction Std Err  Check-Loss Residual
------  --------------  ---------------  ----

NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Interpreting the results

**The tail tells a different story than the center.** Comparing the two coefficient blocks (Step 2) and the merged table (Step 3), the slopes for `ToolWear`, `CycleSpeed`, and `FeedRate` are markedly larger at the 90th percentile than at the median. That is the data-generating mechanism made visible: because we built the error spread to grow with wear and speed, those drivers barely shift the median deviation but strongly inflate the scrap-prone upper tail. A mean-based regression would have under-weighted exactly the levers that matter for scrap.

**The `ToolWear` signal is the clearest.** Its slope roughly triples from the median fit (0.015) to the tail fit (0.042), and the 90th-percentile confidence band sits well above zero — wear is the single most reliable driver of high-tail risk. `CycleSpeed`, essentially flat at the median, turns positive in the tail, consistent with its role in the spread term.

**Quantile regression separates location from spread.** `CoolantTemp` enters the location term but not the spread term, so its slope stays close to 0.4–0.5 microns per degree at both quantiles — it shifts the whole distribution rather than fanning the tail. `Humidity` carries no real signal in the data-generating process, yet on only 100 parts it picks up a small apparent slope; its `Tail - Median` change (0.013) is an order of magnitude smaller than `ToolWear`'s (0.027) and dwarfed by `FeedRate`'s (12.3), so it does not behave like a tail driver. The honest lesson is that a true-null variable can still look non-zero in a small sample — which is exactly why a licensed full-production run over thousands of parts would tighten these estimates.

**Operational payoff.** The `OUTPUT` predictions give every part a predicted 90th-percentile deviation with a standard error, flagging high-tail-risk parts before they ship. The actionable conclusion: tighten tool-change intervals and cap cycle speed when running tight-tolerance jobs, because both controls act directly on the scrap-driving upper tail of dimensional deviation.

*Note on scale:* this notebook runs under the unlicensed engine, which caps each step at 100 observations, so the slopes above are estimated on 100 parts and the tail fit is necessarily noisier than a full-production run would be. The center-versus-tail pattern is already visible at this size; a licensed run over the full part stream would tighten every confidence band.